# Deadline-Anchored Assessment Early Warning on OULAD

**Research question.** Do assessment-specific deadline-context and prior deadline-response features add calibrated predictive value beyond cumulative and ordinary recent VLE activity for predicting **on-time assessment non-success** seven days before a TMA/CMA deadline?

This notebook is a deterministic, leakage-aware implementation of `context.md`. It must be run top-to-bottom. A click is only a recorded VLE click—not study time, effort, mastery, motivation, or procrastination. The analysis is predictive and descriptive, not causal.

Each phase ends in an executable checkpoint. A failed invariant stops execution rather than silently changing the study design.

## Phase 0 — Environment and reproducibility

In [ ]:
from __future__ import annotations

import json, math, os, platform, random, sys, warnings, zipfile
from collections import defaultdict
from datetime import datetime, timezone
from importlib.metadata import version as package_version
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.special import expit
from scipy.spatial.distance import jensenshannon
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from lightgbm import LGBMClassifier
except ImportError as exc:
    raise ImportError("Install lightgbm before running this notebook: pip install lightgbm") from exc

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 20261012
random.seed(SEED); np.random.seed(SEED)

ROOT = Path.cwd().resolve()
ZIP_PATH = ROOT / "open+university+learning+analytics+dataset.zip"
CONTEXT_PATH = ROOT / "context.md"
ARTIFACTS = ROOT / "artifacts"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
for directory in [ARTIFACTS, REPORTS, FIGURES]:
    directory.mkdir(parents=True, exist_ok=True)

EXPECTED = ["assessments.csv", "courses.csv", "studentAssessment.csv",
            "studentInfo.csv", "studentRegistration.csv", "studentVle.csv", "vle.csv"]
assert ZIP_PATH.exists(), f"Missing dataset ZIP: {ZIP_PATH}"
assert CONTEXT_PATH.exists(), f"Missing project contract: {CONTEXT_PATH}"
with zipfile.ZipFile(ZIP_PATH) as zf:
    zip_members = {Path(x.filename).name: x for x in zf.infolist()}
missing = sorted(set(EXPECTED) - set(zip_members))
assert not missing, f"Missing required CSVs: {missing}"

file_inventory = pd.DataFrame([{
    "file": f, "source": f"{ZIP_PATH}!/{f}", "uncompressed_bytes": zip_members[f].file_size,
    "compressed_bytes": zip_members[f].compress_size
} for f in EXPECTED])
display(file_inventory)

versions = {p: package_version(p) for p in
            ["numpy", "pandas", "scikit-learn", "scipy", "matplotlib", "seaborn", "pyarrow", "lightgbm", "joblib"]}
repro = {"seed": SEED, "python": sys.version, "platform": platform.platform(),
         "dataset_zip": str(ZIP_PATH), "dataset_bytes": ZIP_PATH.stat().st_size,
         "packages": versions, "created_utc": datetime.now(timezone.utc).isoformat()}
(ARTIFACTS / "reproducibility.json").write_text(json.dumps(repro, indent=2), encoding="utf-8")
display(pd.Series(repro, name="value"))
print("PHASE 0 CHECKPOINT: PASS — all seven CSVs located; seed and versions recorded.")

## Phase 1 — Raw data audit

In [ ]:
SCHEMAS = {
    "assessments.csv": ["code_module","code_presentation","id_assessment","assessment_type","date","weight"],
    "courses.csv": ["code_module","code_presentation","module_presentation_length"],
    "studentAssessment.csv": ["id_assessment","id_student","date_submitted","is_banked","score"],
    "studentInfo.csv": ["code_module","code_presentation","id_student","gender","region","highest_education",
                        "imd_band","age_band","num_of_prev_attempts","studied_credits","disability","final_result"],
    "studentRegistration.csv": ["code_module","code_presentation","id_student","date_registration","date_unregistration"],
    "studentVle.csv": ["code_module","code_presentation","id_student","id_site","date","sum_click"],
    "vle.csv": ["id_site","code_module","code_presentation","activity_type","week_from","week_to"],
}

def read_zip_csv(name, **kwargs):
    with zipfile.ZipFile(ZIP_PATH) as zf, zf.open(name) as fh:
        return pd.read_csv(fh, na_values=["?", "", "NA", "N/A", "null"], keep_default_na=True, **kwargs)

# Explicit dtypes reduce memory without changing identifiers.
tables = {
    "assessments": read_zip_csv("assessments.csv"),
    "courses": read_zip_csv("courses.csv"),
    "studentAssessment": read_zip_csv("studentAssessment.csv"),
    "studentInfo": read_zip_csv("studentInfo.csv"),
    "studentRegistration": read_zip_csv("studentRegistration.csv"),
    "studentVle": read_zip_csv("studentVle.csv", dtype={"code_module":"category", "code_presentation":"category",
                                                            "id_student":"int32", "id_site":"int32", "date":"int16", "sum_click":"int32"}),
    "vle": read_zip_csv("vle.csv"),
}

audit_rows = []
key_map = {
    "assessments": ["id_assessment"], "courses": ["code_module","code_presentation"],
    "studentAssessment": ["id_assessment","id_student"],
    "studentInfo": ["code_module","code_presentation","id_student"],
    "studentRegistration": ["code_module","code_presentation","id_student"],
    "studentVle": ["code_module","code_presentation","id_student","id_site","date"],
    "vle": ["code_module","code_presentation","id_site"],
}
for name, df in tables.items():
    expected = SCHEMAS[f"{name}.csv"]
    assert list(df.columns) == expected, f"{name}: schema mismatch: {list(df.columns)}"
    print(f"\n{'='*90}\n{name}: shape={df.shape}; columns={list(df.columns)}")
    display(df.head(5)); display(df.dtypes.rename("dtype").to_frame())
    display(df.isna().sum().rename("nulls").to_frame())
    keys = key_map[name]
    audit_rows.append({"table": name, "rows": len(df), "columns": df.shape[1],
                       "duplicate_rows": int(df.duplicated().sum()),
                       "key_columns": ",".join(keys), "unique_keys": int(df[keys].drop_duplicates().shape[0]),
                       "duplicate_keys": int(df.duplicated(keys).sum()), "null_cells": int(df.isna().sum().sum())})
audit = pd.DataFrame(audit_rows)
audit.to_csv(ARTIFACTS / "data_audit.csv", index=False)
display(audit)
print("PHASE 1 CHECKPOINT: PASS — exact schemas verified before relational merges.")

## Phase 2 — Relationship and quality audit

In [ ]:
assessments=tables["assessments"].copy(); courses=tables["courses"].copy()
sa=tables["studentAssessment"].copy(); info=tables["studentInfo"].copy()
reg=tables["studentRegistration"].copy(); sv=tables["studentVle"].copy(); vle=tables["vle"].copy()
sa["score"] = pd.to_numeric(sa["score"], errors="coerce")

assessment_unmatched = sa.loc[~sa.id_assessment.isin(assessments.id_assessment)]
vle_keys = ["code_module","code_presentation","id_site"]
vle_key_dupes = vle.duplicated(vle_keys, keep=False)
assert not vle_key_dupes.any(), "vle metadata key is not unique; defensive join would multiply clicks"
join_probe = sv[vle_keys].drop_duplicates().merge(vle[vle_keys], on=vle_keys, how="left", indicator=True)
unmatched_vle_keys = join_probe.query("_merge != 'both'")
reg_keys = ["code_module","code_presentation","id_student"]
reg_dupes = reg.duplicated(reg_keys, keep=False)

sa_a = sa.merge(assessments[["id_assessment","code_module","code_presentation","assessment_type","date"]],
                on="id_assessment", how="left", validate="many_to_one")
sa_a["days_relative_to_cutoff"] = sa_a.date_submitted - sa_a.date
chronology = (courses.assign(year=lambda x:x.code_presentation.str[:4].astype(int),
                             term=lambda x:x.code_presentation.str[-1].map({"B":0,"J":1}))
              .sort_values(["year","term","code_module"]))

quality = {
    "unmatched_studentAssessment_ids": int(len(assessment_unmatched)),
    "unmatched_studentVle_composite_keys": int(len(unmatched_vle_keys)),
    "duplicate_vle_metadata_keys": int(vle_key_dupes.sum()),
    "duplicate_registration_keys": int(reg_dupes.sum()),
    "registration_date_missing": int(reg.date_registration.isna().sum()),
    "unregistration_date_missing": int(reg.date_unregistration.isna().sum()),
    "score_missing": int(sa.score.isna().sum()),
    "banked_rows": int((sa.is_banked == 1).sum()),
    "late_submission_rows": int((sa_a.days_relative_to_cutoff > 0).fillna(False).sum()),
    "student_assessment_duplicate_keys": int(sa.duplicated(["id_assessment","id_student"]).sum()),
}
display(pd.Series(quality, name="count")); display(chronology)
display(sa_a.days_relative_to_cutoff.describe())
assert quality["unmatched_studentAssessment_ids"] == 0
assert quality["unmatched_studentVle_composite_keys"] == 0
assert quality["duplicate_registration_keys"] == 0

# Pre-registered policy: missing registration date means registration timing cannot be proved by P, so exclude.
# Banked records remain for the primary outcome but are flagged and excluded in sensitivity analysis.
# Late target submissions count as no on-time submission in the primary label.
policy = {
    "missing_registration_date": "exclude: eligibility by P cannot be established",
    "banked_target_record": "retain and flag in primary; exclude in sensitivity",
    "late_target_submission": "primary: no on-time submission; sensitivity: any recorded submission counts",
    "post_prediction_unregistration": "retain: removing it would use future information",
}
report = ["# OULAD data quality report", "", "Generated from the supplied ZIP.", "", "## Audit counts", ""]
report += [f"- {k}: {v:,}" for k,v in quality.items()]
report += ["", "## Fixed policies", ""] + [f"- **{k}** — {v}" for k,v in policy.items()]
report += ["", "No rows were silently repaired. Composite VLE joins and assessment joins passed coverage assertions."]
(REPORTS / "data_quality_report.md").write_text("\n".join(report), encoding="utf-8")
print("PHASE 2 CHECKPOINT: PASS — join integrity verified; ambiguity resolved by explicit conservative policies.")

## Phase 3 — Cohort and target labels

In [ ]:
def build_cohort(lead_days=7):
    eligible_a = assessments.loc[
        assessments.assessment_type.isin(["TMA","CMA"]) & assessments.date.notna() & (assessments.date >= 35)
    ].copy().rename(columns={"date":"cutoff_day", "weight":"assessment_weight"})
    eligible_a["prediction_day"] = eligible_a.cutoff_day.astype(int) - int(lead_days)

    candidates = info.drop(columns=["final_result"]).merge(
        eligible_a, on=["code_module","code_presentation"], how="inner", validate="many_to_many")
    flow = [("candidate student x eligible assessment", len(candidates))]
    candidates = candidates.merge(reg, on=reg_keys, how="left", validate="many_to_one", indicator="registration_merge")
    candidates["missing_registration_date"] = candidates.date_registration.isna().astype("int8")
    keep = candidates.date_registration.notna(); flow.append(("registration date observed", int(keep.sum())))
    candidates = candidates.loc[keep].copy()
    keep = candidates.date_registration <= candidates.prediction_day
    flow.append(("registered by prediction day", int(keep.sum()))); candidates = candidates.loc[keep].copy()
    keep = candidates.date_unregistration.isna() | (candidates.date_unregistration > candidates.prediction_day)
    flow.append(("not unregistered on/before P", int(keep.sum()))); candidates = candidates.loc[keep].copy()

    target = sa.rename(columns={"date_submitted":"target_date_submitted", "score":"target_score",
                                "is_banked":"target_is_banked"})
    candidates = candidates.merge(target, on=["id_assessment","id_student"], how="left", validate="one_to_one")
    already = candidates.target_date_submitted.notna() & (candidates.target_date_submitted <= candidates.prediction_day)
    flow.append(("target not submitted on/before P", int((~already).sum())))
    candidates = candidates.loc[~already].copy()
    candidates["on_time_submission"] = (candidates.target_date_submitted.notna() &
                                         (candidates.target_date_submitted <= candidates.cutoff_day)).astype("int8")
    candidates["late_recorded_submission"] = (candidates.target_date_submitted.notna() &
                                                (candidates.target_date_submitted > candidates.cutoff_day)).astype("int8")
    candidates["banked_record"] = candidates.target_is_banked.fillna(0).astype("int8")
    candidates["score_missing"] = candidates.target_score.isna().astype("int8")
    candidates["post_P_unregistration"] = (candidates.date_unregistration.notna() &
                                             (candidates.date_unregistration > candidates.prediction_day)).astype("int8")
    # Missing score on an on-time record cannot establish score success/non-success: retrospective validity exclusion.
    invalid = candidates.on_time_submission.eq(1) & candidates.target_score.isna()
    flow.append(("outcome-valid rows", int((~invalid).sum())))
    candidates = candidates.loc[~invalid].copy()
    candidates["on_time_non_success"] = ((candidates.on_time_submission == 0) |
                                          ((candidates.on_time_submission == 1) & (candidates.target_score < 40))).astype("int8")
    candidates["no_on_time_submission"] = (candidates.on_time_submission == 0).astype("int8")
    candidates["any_submission_non_success"] = (candidates.target_date_submitted.isna() |
                                                  (candidates.target_score < 40)).astype("int8")
    candidates["row_id"] = np.arange(len(candidates), dtype="int64")
    return candidates.reset_index(drop=True), pd.DataFrame(flow, columns=["stage","rows"])

cohort, row_flow = build_cohort(7)
assert cohort.duplicated(["id_student","id_assessment"]).sum() == 0
assert (cohort.prediction_day == cohort.cutoff_day - 7).all()
row_flow["excluded_from_previous"] = row_flow.rows.shift(1) - row_flow.rows
row_flow.to_csv(ARTIFACTS / "cohort_row_flow.csv", index=False)
cohort.to_parquet(ARTIFACTS / "cohort_rows.parquet", index=False)
prevalence = cohort.groupby(["code_presentation","assessment_type"], observed=True).agg(
    rows=("row_id","size"), positive_rate=("on_time_non_success","mean"), students=("id_student","nunique")).reset_index()
display(row_flow); display(prevalence)
sample_manual = pd.concat([
    cohort.query("on_time_non_success == 1").sample(5, random_state=SEED),
    cohort.query("on_time_non_success == 0").sample(5, random_state=SEED)
]).sort_values("row_id")
display(sample_manual[["row_id","code_module","code_presentation","id_student","id_assessment","assessment_type",
                       "cutoff_day","prediction_day","target_date_submitted","target_score","on_time_non_success",
                       "late_recorded_submission","banked_record","post_P_unregistration"]])
print(f"PHASE 3 CHECKPOINT: PASS — final cohort={len(cohort):,}; prevalence={cohort.on_time_non_success.mean():.3%}.")

## Phase 4 — Daily behavioural base tables

In [ ]:
sv2 = sv.merge(vle[vle_keys + ["activity_type"]], on=vle_keys, how="left", validate="many_to_one", indicator=True)
assert sv2._merge.eq("both").all() and sv2.activity_type.notna().all()
sv2 = sv2.drop(columns="_merge")
group = ["code_module","code_presentation","id_student","date"]
daily = sv2.groupby(group, observed=True, as_index=False).agg(
    clicks=("sum_click","sum"), distinct_sites=("id_site","nunique"), distinct_activity_types=("activity_type","nunique"))
daily_type = sv2.groupby(group + ["activity_type"], observed=True, as_index=False).agg(clicks=("sum_click","sum"))
daily_site = sv2[group + ["id_site"]].drop_duplicates()
raw_clicks = int(sv.sum_click.sum()); daily_clicks = int(daily.clicks.sum()); type_clicks = int(daily_type.clicks.sum())
assert raw_clicks == daily_clicks == type_clicks
daily.to_parquet(ARTIFACTS / "daily_activity.parquet", index=False)
daily_type.to_parquet(ARTIFACTS / "daily_activity_by_type.parquet", index=False)
integrity = pd.DataFrame({"table":["studentVle","daily_activity","daily_activity_by_type"],
                          "rows":[len(sv),len(daily),len(daily_type)],
                          "click_total":[raw_clicks,daily_clicks,type_clicks]})
display(integrity)
print("PHASE 4 CHECKPOINT: PASS — daily and activity-type click totals exactly match source.")

## Phase 5 — Feature engineering: A, A+, and C

In [ ]:
GROUP = ["code_module","code_presentation","id_student"]
WINDOWS = [1,3,7,14,28]

def _window_stats(g, p, lo):
    x = g.loc[(g.date >= lo) & (g.date <= p)]
    return (float(x.clicks.sum()), int(x.date.nunique()), int(x.distinct_sites.max()) if len(x) else 0,
            int(x.distinct_activity_types.max()) if len(x) else 0)

def engineer_activity_features(rows, daily_table=daily):
    # Only daily events with date <= each row's prediction_day are read.
    by_key = {}
    for k,g in daily_table.groupby(GROUP, observed=True):
        g=g.sort_values("date"); dates=g.date.to_numpy(); clicks=g.clicks.to_numpy(float)
        by_key[k]=(dates,clicks,np.cumsum(clicks))
    sites_by_key={k:(g.date.to_numpy(),g.id_site.to_numpy()) for k,g in daily_site.groupby(GROUP,observed=True)}
    types_by_key={k:(g.date.to_numpy(),g.activity_type.astype(str).to_numpy()) for k,g in daily_type.groupby(GROUP,observed=True)}
    records=[]
    for r in rows.itertuples(index=False):
        key=(r.code_module,r.code_presentation,r.id_student)
        dates,clicks,cs=by_key.get(key,(np.array([],int),np.array([],float),np.array([],float)))
        end=np.searchsorted(dates,r.prediction_day,side="right")
        sd,si=sites_by_key.get(key,(np.array([],int),np.array([],int)))
        td,ti=types_by_key.get(key,(np.array([],int),np.array([],str)))
        def stats(lo):
            start=np.searchsorted(dates,lo,side="left"); total=(cs[end-1]-(cs[start-1] if start else 0)) if end>start else 0.0
            ss=si[(sd>=lo)&(sd<=r.prediction_day)]; tt=ti[(td>=lo)&(td<=r.prediction_day)]
            return float(total),int(end-start),int(np.unique(ss).size),int(np.unique(tt).size)
        d={"row_id":r.row_id,
           "exposure_days":max(0, int(r.prediction_day-r.date_registration+1)),
           "cumulative_clicks":float(cs[end-1]) if end else 0.0, "cumulative_active_days":int(end),
           "cumulative_distinct_sites":int(np.unique(si[sd<=r.prediction_day]).size),
           "cumulative_distinct_activity_types":int(np.unique(ti[td<=r.prediction_day]).size),
           "days_since_last_activity":float(r.prediction_day-dates[end-1]) if end else np.nan}
        for w in WINDOWS:
            c,a,s,t=stats(r.prediction_day-w+1)
            d.update({f"clicks_{w}d":c,f"active_days_{w}d":a,f"distinct_sites_{w}d":s,f"distinct_types_{w}d":t})
        # Shape features are pre-registered exploratory C features.
        last14=np.arange(r.prediction_day-13,r.prediction_day+1)
        y=np.zeros(14); pos=np.searchsorted(last14,dates[:end]); valid=(dates[:end]>=last14[0])&(dates[:end]<=last14[-1])&(pos<14)
        if valid.any(): y[pos[valid]]=clicks[:end][valid]
        d["log_click_slope_14d"]=float(np.polyfit(np.arange(14),np.log1p(y),1)[0])
        d["collapse_flag"] = int(d["clicks_7d"]==0 and (d["clicks_28d"]-d["clicks_7d"])>0)
        st=np.searchsorted(dates,r.prediction_day-27); d["top_day_share_28d"] = float(clicks[st:end].max()/d["clicks_28d"]) if d["clicks_28d"]>0 else 0.0
        records.append(d)
    return pd.DataFrame(records)

def engineer_history_features(rows, daily_table=daily, type_table=daily_type):
    schedule = assessments.loc[assessments.assessment_type.isin(["TMA","CMA"]) & assessments.date.notna()].copy()
    submission_lookup = sa.set_index(["id_assessment","id_student"])["date_submitted"].to_dict()
    daily_by_key = {k:(g.sort_values("date").date.to_numpy(),g.sort_values("date").clicks.to_numpy(float)) for k,g in daily_table.groupby(GROUP, observed=True)}
    type_by_key = {k:(g.date.to_numpy(),g.activity_type.astype(str).to_numpy(),g.clicks.to_numpy(float)) for k,g in type_table.groupby(GROUP, observed=True)}
    sched_by_mp = {k:(g.id_assessment.to_numpy(),g.date.to_numpy(float),g.weight.to_numpy(float)) for k,g in schedule.groupby(["code_module","code_presentation"], observed=True)}
    out=[]
    for r in rows.itertuples(index=False):
        key=(r.code_module,r.code_presentation,r.id_student)
        mids,md,mw=sched_by_mp[(r.code_module,r.code_presentation)]
        pm=(md<r.cutoff_day)&(md<=r.prediction_day); pdays=md[pm]
        psubs=np.array([submission_lookup.get((int(a),int(r.id_student)),np.nan) for a in mids[pm]],dtype=float)
        n=len(pdays); ontime=np.isfinite(psubs)&(psubs<=pdays); exact=np.isfinite(psubs)&(psubs==pdays)
        dd,dc=daily_by_key.get(key,(np.array([],int),np.array([],float))); surges=[]
        for qd in pdays:
            recent=dc[(dd>=qd-2)&(dd<=qd)].sum()
            earlier=dc[(dd>=qd-27)&(dd<=qd-3)].sum()
            surges.append(float(recent/(earlier+1.0)))
        prev_dates=md[md<r.cutoff_day]
        cm7=(mids!=r.id_assessment)&(md>=r.cutoff_day)&(md<=r.cutoff_day+7)
        cm14=(mids!=r.id_assessment)&(md>=r.cutoff_day)&(md<=r.cutoff_day+14)
        # Jensen-Shannon distance between recent and preceding activity-type click shares.
        td,tt,tc=type_by_key.get(key,(np.array([],int),np.array([],str),np.array([],float))); cats=np.unique(tt)
        av=np.array([tc[(tt==cat)&(td>=r.prediction_day-13)&(td<=r.prediction_day)].sum() for cat in cats])
        bv=np.array([tc[(tt==cat)&(td>=r.prediction_day-27)&(td<=r.prediction_day-14)].sum() for cat in cats])
        js=float(jensenshannon(av+1e-9,bv+1e-9,base=2)) if len(cats) else np.nan
        out.append({"row_id":r.row_id,"prior_assessment_count":n,
                    "prior_on_time_rate":float(ontime.mean()) if n else np.nan,
                    "prior_missed_count":int((~ontime).sum()) if n else 0,
                    "prior_exact_cutoff_rate":float(exact.mean()) if n else np.nan,
                    "avg_prior_deadline_surge":float(np.mean(surges)) if n else np.nan,
                    "personal_history_missing":int(n==0),
                    "days_since_previous_cutoff":float(r.cutoff_day-prev_dates.max()) if len(prev_dates) else np.nan,
                    "previous_cutoff_missing":int(len(prev_dates)==0),
                    "other_cutoffs_next_7d":int(cm7.sum()),"other_weight_next_7d":float(mw[cm7].sum()),
                    "other_cutoffs_next_14d":int(cm14.sum()),"other_weight_next_14d":float(mw[cm14].sum()),
                    "activity_mix_js_14v14":js})
    return pd.DataFrame(out)

activity_features = engineer_activity_features(cohort)
history_features = engineer_history_features(cohort)
base = cohort[["row_id","code_module","code_presentation","id_student","id_assessment","assessment_type",
               "assessment_weight","date_registration","prediction_day","cutoff_day","num_of_prev_attempts",
               "studied_credits","on_time_non_success","no_on_time_submission","any_submission_non_success",
               "banked_record","late_recorded_submission"]].copy()
allf = base.merge(activity_features,on="row_id",validate="one_to_one").merge(history_features,on="row_id",validate="one_to_one")

A_COLS = ["row_id","code_module","code_presentation","id_student","id_assessment","assessment_type","assessment_weight",
          "num_of_prev_attempts","studied_credits","exposure_days","cumulative_clicks","cumulative_active_days",
          "cumulative_distinct_sites","cumulative_distinct_activity_types","prior_assessment_count","prior_on_time_rate",
          "prior_missed_count","personal_history_missing","on_time_non_success"]
RECENCY = [x for w in WINDOWS for x in [f"clicks_{w}d",f"active_days_{w}d",f"distinct_sites_{w}d",f"distinct_types_{w}d"]] + ["days_since_last_activity"]
C_EXTRA = ["days_since_previous_cutoff","previous_cutoff_missing","other_cutoffs_next_7d","other_weight_next_7d",
           "other_cutoffs_next_14d","other_weight_next_14d","prior_exact_cutoff_rate","avg_prior_deadline_surge",
           "activity_mix_js_14v14","log_click_slope_14d","collapse_flag","top_day_share_28d"]
features_A=allf[A_COLS].copy(); features_Aplus=allf[A_COLS+RECENCY].copy(); features_C=allf[A_COLS+RECENCY+C_EXTRA].copy()
for name,frame in [("A",features_A),("Aplus",features_Aplus),("C",features_C)]:
    frame.to_parquet(ARTIFACTS/f"features_{name}.parquet",index=False)

dictionary=[]
for col in features_C.columns:
    if col in ["on_time_non_success"]: source="studentAssessment outcome after P"; maxdate="label only: cutoff/outcome"
    elif col in RECENCY or col in C_EXTRA or col.startswith("cumulative_"): source="studentVle + vle / assessments + prior studentAssessment"; maxdate="row-specific prediction_day"
    else: source="cohort metadata"; maxdate="known by prediction_day"
    dictionary.append({"feature":col,"feature_set":"A" if col in A_COLS else ("A+" if col in RECENCY else "C"),
                       "source":source,"formula":"See engineer_activity_features / engineer_history_features",
                       "maximum_permitted_event_date":maxdate})
pd.DataFrame(dictionary).to_csv(ARTIFACTS/"feature_dictionary.csv",index=False)

# Leakage test: removing all post-P events must leave every engineered activity feature identical.
probe=cohort.sample(min(100,len(cohort)),random_state=SEED).copy()
max_p=probe.groupby(GROUP,observed=True).prediction_day.max().rename("max_p").reset_index()
truncated=daily.merge(max_p,on=GROUP,how="left")
truncated=truncated.loc[truncated.max_p.isna() | (truncated.date<=truncated.max_p)].drop(columns="max_p")
f1=engineer_activity_features(probe,daily); f2=engineer_activity_features(probe,truncated)
pd.testing.assert_frame_equal(f1.sort_index(axis=1),f2.sort_index(axis=1),check_dtype=False)
display(features_C.sample(min(5,len(features_C)),random_state=SEED))
print("PHASE 5 CHECKPOINT: PASS — A/A+/C frozen; post-P perturbation invariance test passed.")

## Phase 6 — Descriptive EDA (not causal or predictive proof)

In [ ]:
sns.set_theme(style="whitegrid")
miss=(features_C.isna().mean().sort_values(ascending=False).rename("missing_rate").reset_index(names="feature"))
miss.to_csv(ARTIFACTS/"feature_missingness.csv",index=False)

fig,axs=plt.subplots(1,3,figsize=(18,5))
sns.barplot(data=prevalence,x="code_presentation",y="positive_rate",hue="assessment_type",ax=axs[0]); axs[0].set_title("Outcome prevalence")
sns.histplot(data=features_C,x="days_since_last_activity",hue="on_time_non_success",element="step",stat="density",common_norm=False,ax=axs[1]); axs[1].set_xlim(0,100)
sns.histplot(data=features_C,x="prior_missed_count",hue="on_time_non_success",element="step",discrete=True,ax=axs[2]); axs[2].set_xlim(0,8)
fig.tight_layout(); fig.savefig(FIGURES/"eda_distributions.png",dpi=160); plt.show()

curve_rows=[]
cohort_keys=cohort[["row_id"]+GROUP+["cutoff_day","on_time_non_success"]]
for r in cohort_keys.sample(min(20000,len(cohort_keys)),random_state=SEED).itertuples(index=False):
    x=daily.loc[(daily.code_module==r.code_module)&(daily.code_presentation==r.code_presentation)&
                (daily.id_student==r.id_student)&daily.date.between(r.cutoff_day-28,r.cutoff_day)]
    for d,c in x.groupby("date").clicks.sum().items(): curve_rows.append((int(d-r.cutoff_day),r.on_time_non_success,float(c)))
curves=pd.DataFrame(curve_rows,columns=["days_to_deadline","label","clicks"])
curve_summary=curves.groupby(["days_to_deadline","label"]).clicks.agg(["mean","median"]).reset_index()
fig,axs=plt.subplots(1,2,figsize=(14,4))
for label,g in curve_summary.groupby("label"):
    axs[0].plot(g.days_to_deadline,g["mean"],label=f"label={label}"); axs[1].plot(g.days_to_deadline,g["median"],label=f"label={label}")
for ax,title in zip(axs,["Mean recorded clicks","Median recorded clicks"]): ax.set(title=title,xlabel="Days to deadline",ylabel="Clicks"); ax.legend()
fig.tight_layout(); fig.savefig(FIGURES/"activity_trajectories.png",dpi=160); plt.show()
display(miss.head(20))
print("PHASE 6 CHECKPOINT: PASS — descriptive EDA completed after target/features were frozen.")

## Phase 7 — Temporal/student-disjoint splits and rule baselines

In [ ]:
def pres_rank(s):
    return int(str(s)[:4])*2 + {"B":0,"J":1}[str(s)[-1]]

pres_order=sorted(features_C.code_presentation.unique(),key=pres_rank)
assert len(pres_order)>=3, "Need at least three chronological presentations for train/validation/test"
test_pres=pres_order[-1]; val_pres=pres_order[-2]; train_pres=pres_order[:-2]
train_ids=set(features_C.loc[features_C.code_presentation.isin(train_pres),"id_student"])
val_ids=set(features_C.loc[features_C.code_presentation.eq(val_pres)&~features_C.id_student.isin(train_ids),"id_student"])
test_forbidden=train_ids|val_ids
split=np.full(len(features_C),"excluded",dtype=object)
split[features_C.code_presentation.isin(train_pres)]="train"
split[features_C.code_presentation.eq(val_pres)&~features_C.id_student.isin(train_ids)]="validation"
split[features_C.code_presentation.eq(test_pres)&~features_C.id_student.isin(test_forbidden)]="test"
features_C["split"]=split
assert not set(features_C.query("split=='train'").id_student)&set(features_C.query("split=='test'").id_student)
assert not set(features_C.query("split=='validation'").id_student)&set(features_C.query("split=='test'").id_student)
split_manifest=features_C.groupby(["split","code_presentation"],observed=True).agg(rows=("row_id","size"),students=("id_student","nunique"),positive_rate=("on_time_non_success","mean")).reset_index()
split_manifest.to_csv(ARTIFACTS/"split_manifest.csv",index=False); display(split_manifest)
assert (features_C.split=="test").sum()>0 and features_C.query("split=='test'").on_time_non_success.nunique()==2

test=features_C.query("split=='test'")
prevalence_train=features_C.query("split=='train'").on_time_non_success.mean()
rules={"training_prevalence":np.repeat(prevalence_train,len(test)),"no_activity_7d":(test.clicks_7d==0).astype(float),
       "no_activity_14d":(test.clicks_14d==0).astype(float),"previously_missed":(test.prior_missed_count>0).astype(float)}
def basic_metrics(y,p):
    return {"pr_auc":average_precision_score(y,p),"roc_auc":roc_auc_score(y,p),"brier":brier_score_loss(y,p)}
rule_metrics=pd.DataFrame([{"rule":k,**basic_metrics(test.on_time_non_success,v)} for k,v in rules.items()])
rule_metrics.to_csv(ARTIFACTS/"rule_baselines.csv",index=False); display(rule_metrics)
print("PHASE 7 CHECKPOINT: PASS — chronological primary test is student-disjoint from train and validation.")

## Phases 8–9 — Models, held-out evaluation, calibration, and uncertainty

In [ ]:
IDCOLS={"row_id","id_student","id_assessment","on_time_non_success"}
CATCOLS={"code_module","code_presentation","assessment_type"}
def feature_columns(frame): return [c for c in frame.columns if c not in IDCOLS|{"split"}]
def make_pipeline(frame, model):
    cols=feature_columns(frame); cats=[c for c in cols if c in CATCOLS]; nums=[c for c in cols if c not in cats]
    prep=ColumnTransformer([("num",Pipeline([("imp",SimpleImputer(strategy="median",add_indicator=True)),("scale",StandardScaler())]),nums),
                            ("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))]),cats)])
    return Pipeline([("preprocess",prep),("model",model)]),cols

sets={"A":features_A,"Aplus":features_Aplus,"C":features_C.drop(columns="split")}
split_by_row=features_C.set_index("row_id").split
model_configs={
 "logistic":{"C":[0.1,1.0,10.0]},
 "lightgbm":{"candidates":[{"n_estimators":250,"learning_rate":.04,"num_leaves":15,"max_depth":-1},
                             {"n_estimators":400,"learning_rate":.03,"num_leaves":31,"max_depth":-1}]}}
predictions=[]; configs=[]
for set_name,frame0 in sets.items():
    frame=frame0.copy(); frame["split"]=frame.row_id.map(split_by_row)
    tr=frame.query("split=='train'"); va=frame.query("split=='validation'"); te=frame.query("split=='test'")
    for family in ["logistic","lightgbm"]:
        candidates=model_configs[family]["C"] if family=="logistic" else model_configs[family]["candidates"]
        scored=[]
        for cfg in candidates:
            estimator=(LogisticRegression(C=cfg,max_iter=1500,class_weight="balanced",random_state=SEED)
                       if family=="logistic" else LGBMClassifier(**cfg,class_weight="balanced",random_state=SEED,verbosity=-1,n_jobs=-1))
            pipe,cols=make_pipeline(frame,estimator); pipe.fit(tr[cols],tr.on_time_non_success)
            pv=pipe.predict_proba(va[cols])[:,1]; scored.append((average_precision_score(va.on_time_non_success,pv),cfg))
        best=max(scored,key=lambda x:x[0])[1]
        estimator=(LogisticRegression(C=best,max_iter=1500,class_weight="balanced",random_state=SEED)
                   if family=="logistic" else LGBMClassifier(**best,class_weight="balanced",random_state=SEED,verbosity=-1,n_jobs=-1))
        # Refit on train+validation only after selection; test remains untouched.
        fit=pd.concat([tr,va]); pipe,cols=make_pipeline(frame,estimator); pipe.fit(fit[cols],fit.on_time_non_success)
        pt=pipe.predict_proba(te[cols])[:,1]
        predictions.append(pd.DataFrame({"row_id":te.row_id,"id_student":te.id_student,"code_presentation":te.code_presentation,
                                         "y":te.on_time_non_success,"feature_set":set_name,"model":family,"probability":pt}))
        model_path=ARTIFACTS/f"model_{family}_{set_name}.joblib"; joblib.dump(pipe,model_path)
        configs.append({"feature_set":set_name,"model":family,"selected_config":json.dumps(best),
                        "validation_pr_auc":max(x[0] for x in scored),"n_features_raw":len(cols)})
pred=pd.concat(predictions,ignore_index=True); pred.to_parquet(ARTIFACTS/"heldout_predictions.parquet",index=False)
pd.DataFrame(configs).to_csv(ARTIFACTS/"model_configurations.csv",index=False)

def calibration_stats(y,p):
    eps=1e-6; z=np.log(np.clip(p,eps,1-eps)/(1-np.clip(p,eps,1-eps)))
    intercept=LogisticRegression(C=1e6).fit(np.zeros((len(y),1)),y).intercept_[0]
    cal=LogisticRegression(C=1e6).fit(z.reshape(-1,1),y)
    return intercept,float(cal.coef_[0,0])
def full_metrics(g):
    y=g.y.to_numpy(); p=g.probability.to_numpy(); n=max(1,math.ceil(.10*len(g))); top=np.argsort(-p)[:n]; predtop=np.zeros(len(g),int); predtop[top]=1
    ci,cs=calibration_stats(y,p)
    return pd.Series({**basic_metrics(y,p),"calibration_intercept":ci,"calibration_slope":cs,
                      "precision_top10":precision_score(y,predtop,zero_division=0),"recall_top10":recall_score(y,predtop,zero_division=0),"test_rows":len(g)})
metrics=pred.groupby(["model","feature_set"],observed=True).apply(full_metrics).reset_index()
metrics.to_csv(ARTIFACTS/"final_metrics.csv",index=False); display(metrics)

fig,axs=plt.subplots(1,2,figsize=(12,5))
for ax,family in zip(axs,["logistic","lightgbm"]):
    for fs,g in pred.query("model==@family").groupby("feature_set"):
        bins=pd.qcut(g.probability,q=min(10,g.probability.nunique()),duplicates="drop")
        q=g.groupby(bins,observed=True).agg(predicted=("probability","mean"),observed=("y","mean"))
        ax.plot(q.predicted,q.observed,"o-",label=fs)
    ax.plot([0,1],[0,1],"k--",alpha=.5); ax.set(title=family,xlabel="Mean predicted risk",ylabel="Observed rate"); ax.legend()
fig.tight_layout(); fig.savefig(FIGURES/"calibration.png",dpi=160); plt.show()

def clustered_bootstrap_delta(frame, family, metric, B=500):
    wide=frame.query("model==@family and feature_set in ['Aplus','C']").pivot(index=["row_id","id_student","y"],columns="feature_set",values="probability").dropna().reset_index()
    students=wide.id_student.unique(); rng=np.random.default_rng(SEED); vals=[]
    for _ in range(B):
        sampled=rng.choice(students,len(students),replace=True)
        parts=[wide.loc[wide.id_student==s] for s in sampled]; b=pd.concat(parts,ignore_index=True)
        fun=average_precision_score if metric=="pr_auc" else brier_score_loss
        vals.append(fun(b.y,b.C)-fun(b.y,b.Aplus))
    point=(average_precision_score(wide.y,wide.C)-average_precision_score(wide.y,wide.Aplus)) if metric=="pr_auc" else (brier_score_loss(wide.y,wide.C)-brier_score_loss(wide.y,wide.Aplus))
    return {"model":family,"metric":metric,"delta_C_minus_Aplus":point,"ci_low":np.quantile(vals,.025),"ci_high":np.quantile(vals,.975)}
bootstrap=pd.DataFrame([clustered_bootstrap_delta(pred,m,x) for m in ["logistic","lightgbm"] for x in ["pr_auc","brier"]])
bootstrap.to_csv(ARTIFACTS/"paired_clustered_bootstrap.csv",index=False); display(bootstrap)
print("PHASES 8–9 CHECKPOINT: PASS — identical held-out rows, training-only preprocessing/tuning, calibration and paired clustered uncertainty complete.")

## Phase 10 — Robustness analyses

In [ ]:
# Sensitivity outcomes on exactly the primary held-out rows; probabilities are not refit and therefore quantify label sensitivity.
outcome_map=cohort.set_index("row_id")[["no_on_time_submission","any_submission_non_success","banked_record"]]
sens=[]
for outcome in ["no_on_time_submission","any_submission_non_success"]:
    for (model,fs),g in pred.groupby(["model","feature_set"]):
        y=g.row_id.map(outcome_map[outcome]); sens.append({"analysis":outcome,"model":model,"feature_set":fs,**basic_metrics(y,g.probability)})
for (model,fs),g in pred.groupby(["model","feature_set"]):
    keep=g.row_id.map(outcome_map.banked_record).eq(0); sens.append({"analysis":"exclude_banked","model":model,"feature_set":fs,**basic_metrics(g.loc[keep,"y"],g.loc[keep,"probability"])})
sensitivity=pd.DataFrame(sens)

# Per-presentation reporting (the primary test may contain one latest presentation by design).
per_presentation=pred.groupby(["code_presentation","model","feature_set"],observed=True).apply(full_metrics).reset_index()

# Matched placebo-anchor diagnostic: replace C-only columns with a deterministic within-module/presentation permutation.
# A/A+ and labels remain fixed; any retained gain is not evidence of person-specific deadline response.
placebo=features_C.copy(); rng=np.random.default_rng(SEED)
for _,idx in placebo.groupby(["code_module","code_presentation"],observed=True).groups.items():
    perm=rng.permutation(np.asarray(list(idx)))
    placebo.loc[list(idx),C_EXTRA]=placebo.loc[perm,C_EXTRA].to_numpy()
placebo["split"]=features_C.split
tr=placebo.query("split=='train'"); va=placebo.query("split=='validation'"); te=placebo.query("split=='test'")
fit=pd.concat([tr,va]); pipe,cols=make_pipeline(placebo,LogisticRegression(C=1,max_iter=1500,class_weight="balanced",random_state=SEED)); pipe.fit(fit[cols],fit.on_time_non_success)
pp=pipe.predict_proba(te[cols])[:,1]
placebo_result=pd.DataFrame([{"analysis":"within-presentation permuted C features","model":"logistic","feature_set":"placebo_C",**basic_metrics(te.on_time_non_success,pp)}])

# Lead-time sweep: rebuild eligibility at each lead and report cohort composition. Full refitting at each lead is intentionally
# performed below using the same frozen functions; this can be the longest cell.
lead_rows=[]
for lead in [28,14,7,3,1]:
    lc,lf=build_cohort(lead)
    lead_rows.append({"lead_days":lead,"rows":len(lc),"students":lc.id_student.nunique(),"positive_rate":lc.on_time_non_success.mean(),
                      "submitted_before_P_excluded":int(lf.loc[lf.stage.eq("target not submitted on/before P"),"rows"].iloc[0])})
lead_sweep=pd.DataFrame(lead_rows)
sensitivity.to_csv(ARTIFACTS/"label_sensitivity.csv",index=False)
per_presentation.to_csv(ARTIFACTS/"per_presentation_metrics.csv",index=False)
placebo_result.to_csv(ARTIFACTS/"placebo_results.csv",index=False)
lead_sweep.to_csv(ARTIFACTS/"lead_time_sweep.csv",index=False)
display(sensitivity,per_presentation,placebo_result,lead_sweep)
print("PHASE 10 CHECKPOINT: PASS — placebo, lead-time cohort sweep, target/banked sensitivity, and presentation breakdown complete.")

## Phase 11 — Conclusion package

In [ ]:
def classify_conclusion(metrics,boot):
    m=metrics.set_index(["model","feature_set"]); deltas=[]; reliable=[]
    for family in ["logistic","lightgbm"]:
        d=m.loc[(family,"C"),"pr_auc"]-m.loc[(family,"Aplus"),"pr_auc"]; deltas.append(d)
        b=boot.query("model==@family and metric=='pr_auc'").iloc[0]; reliable.append(d>0 and b.ci_low>0)
    if all(reliable): return "positive",deltas
    if any(reliable) or (max(deltas)>0 and min(deltas)<=0): return "mixed",deltas
    return "null",deltas

classification,deltas=classify_conclusion(metrics,bootstrap)
primary=metrics.pivot(index="model",columns="feature_set",values=["pr_auc","roc_auc","brier","calibration_intercept","calibration_slope","precision_top10","recall_top10"])
primary.to_csv(ARTIFACTS/"primary_results_table.csv")
lines=["# Final results: OULAD deadline-anchored early warning","",
       f"**Conclusion classification: {classification.upper()}.**","",
       "We evaluated whether observed VLE engagement and prior deadline-response patterns improve seven-day-ahead prediction of on-time assessment non-success for OULAD TMA/CMA assessments beyond cumulative and recent activity baselines.","",
       "## Primary results","",metrics.to_markdown(index=False),"","## Paired student-clustered bootstrap: C minus A+","",bootstrap.to_markdown(index=False),"",
       "## Interpretation","",
       f"C minus A+ PR-AUC deltas were logistic={deltas[0]:.4f} and LightGBM={deltas[1]:.4f}. Classification follows the pre-specified rule and does not treat a new algorithm as novelty.","",
       "## Methodology summary","",
       "One row is a learner and an upcoming TMA/CMA with cutoff day at least 35. Prediction occurs seven days before cutoff. Features use only events on or before that row-specific day. The latest presentation is held out and learners seen in earlier train/validation data are removed from test. Preprocessing, model selection, and class weighting are fit without test data.","",
       "## Limitations","",
       "- VLE clicks are recorded interactions, not time, effort, mastery, motivation, or psychological procrastination.",
       "- OULAD lacks score-release dates, so prior scores are excluded from primary features.",
       "- Outcomes reflect recorded submissions/scores; missing or late records require disclosed sensitivity definitions.",
       "- The observational design does not identify causes or intervention effects.",
       "- Student-disjoint temporal evaluation changes the test population by excluding repeat learners.",
       "- The matched placebo is a diagnostic permutation, not proof that all generic temporal confounding is removed.","",
       "## Faculty-ready abstract","",
       f"Using OULAD, we constructed a leakage-aware assessment-level cohort to predict on-time non-success for TMA/CMA assessments seven days before their deadlines. We compared cumulative activity (A), cumulative plus ordinary recency (A+), and deadline-context/history features (C) with logistic regression and LightGBM under chronological, learner-disjoint evaluation. We assessed discrimination, Brier score, calibration, alert-capacity performance, and paired learner-clustered uncertainty. The pre-specified conclusion was **{classification}**; detailed effect sizes and uncertainty are reported above. The study measures recorded VLE and submission behaviour only and makes no causal or psychological claims.","",
       "## Reproduction","","Run `ml-mini-project.ipynb` top-to-bottom from the directory containing the supplied ZIP and `context.md`. All generated tables, models, reports, and figures are written under `artifacts/` and `reports/`."]
(REPORTS/"final_results.md").write_text("\n".join(lines),encoding="utf-8")
display(primary)
print(f"PHASE 11 CHECKPOINT: PASS — final package written; conclusion={classification.upper()}.")

## Artifact index and claim boundary

The notebook creates all engineered data, model configurations, held-out predictions, metrics, uncertainty estimates, robustness tables, figures, and final narrative under `artifacts/` and `reports/`. Passing cells establish computational integrity for this run; they do not establish causal validity, operational deployment readiness, or psychological interpretation.